In [1]:
%pip -q install pandas scikit-learn transformers accelerate sentence-transformers

In [2]:
import os, re, json, csv, uuid, shutil, zipfile, unicodedata, difflib
from collections import Counter
from pathlib import Path
from datetime import datetime, date, timedelta
import numpy as np
import pandas as pd

try:
    display
except NameError:
    def display(x): print(x)

REFERENCE_DATE = date(2026, 10, 20)
DATASET_DIR = Path(os.getenv("DATASET_DIR", "/content/Capstone_Project_Dataset_FTTT_2026"))
MODEL_ID    = os.getenv("MODEL_ID", "google/gemma-2-2b-it")
EMBED_MODEL = os.getenv("EMBED_MODEL", "paraphrase-multilingual-MiniLM-L12-v2")  # handles Hindi script
OUTPUT_DIR  = Path(os.getenv("OUTPUT_DIR", "/content/evaluation_outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
zip_path = Path("/content/Capstone_Project_Dataset_FTTT_2026.zip")
if zip_path.is_file() and not DATASET_DIR.is_dir():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(zip_path.parent)

ROOT = DATASET_DIR
KB_DIR, DATA_DIR, PRACTICE_DIR = ROOT / "kb", ROOT / "data", ROOT / "practice"
for p in (KB_DIR, DATA_DIR, PRACTICE_DIR):
    if not p.is_dir():
        raise FileNotFoundError(f"Missing dataset folder: {p}")
TICKETS_FILE = OUTPUT_DIR / "tickets_created.csv"
print("Dataset root:", ROOT)
print("Reference date:", REFERENCE_DATE)

Dataset root: /content/Capstone_Project_Dataset_FTTT_2026
Reference date: 2026-10-20


In [4]:

SYSTEM_PROMPT = """You are ZentroMart's customer-support AI assistant.
Be polite, concise and conversational. Never invent policies, dates, amounts, order status or actions.
For policy answers use only the supplied evidence and say so when the evidence does not answer the question.
Current policy has higher authority than agent guides, help pages, FAQs, marketing, historical logs and superseded policy.
Do not reveal order-specific information before the caller is verified. Never expose verification secrets.
Do not claim a ticket was created unless a tool result confirms it.
The dataset reference date is 20 October 2026. Business days are Monday-Saturday excluding dataset holidays;
return windows use calendar days. Answer in the customer's language when feasible."""

In [5]:
def normalize_digits(text):
    """Convert any Unicode digits (e.g. Devanagari) to ASCII digits."""
    out = []
    for ch in str(text):
        if ch.isdigit():
            try: out.append(str(unicodedata.digit(ch)))
            except (TypeError, ValueError): out.append(ch)
        else:
            out.append(ch)
    return "".join(out)

MONTH_RE = r"(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:t(?:ember)?)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)"
MONTH_NUM = {m: i + 1 for i, m in enumerate("jan feb mar apr may jun jul aug sep oct nov dec".split())}
GROUP_RE = re.compile(r"(?<!\d)\d(?:[ \-]?\d)*(?!\d)")          # "8978 2837" is ONE number
DATE_DMY = re.compile(rf"\b(\d{{1,2}})(?:st|nd|rd|th)?\s+({MONTH_RE})\b\.?,?\s*(\d{{4}})?", re.I)
DATE_MDY = re.compile(rf"\b({MONTH_RE})\.?\s+(\d{{1,2}})(?:st|nd|rd|th)?\b,?\s*(\d{{4}})?", re.I)
DATE_ISO = re.compile(r"\b(\d{4})-(\d{2})-(\d{2})\b")

def _mk_date(y, m, d):
    try: return date(int(y), int(m), int(d))
    except (ValueError, TypeError): return None

def extract_dates(text, default_year=REFERENCE_DATE.year):
    """Explicit dates plus simple relative words, resolved against REFERENCE_DATE."""
    text = normalize_digits(text); found = []
    for m in DATE_ISO.finditer(text):
        d = _mk_date(m.group(1), m.group(2), m.group(3)); d and found.append(d)
    for m in DATE_DMY.finditer(text):
        d = _mk_date(m.group(3) or default_year, MONTH_NUM[m.group(2)[:3].lower()], m.group(1)); d and found.append(d)
    for m in DATE_MDY.finditer(text):
        d = _mk_date(m.group(3) or default_year, MONTH_NUM[m.group(1)[:3].lower()], m.group(2)); d and found.append(d)
    low = text.lower()
    if re.search(r"day before yesterday|\bparso\b|परसों", low): found.append(REFERENCE_DATE - timedelta(days=2))
    elif re.search(r"\byesterday\b", low): found.append(REFERENCE_DATE - timedelta(days=1))
    if re.search(r"\btoday\b|\baaj\b|आज", low): found.append(REFERENCE_DATE)
    out = []
    for d in found:
        if d not in out: out.append(d)
    return out

In [6]:
def load_holidays(extra=()):
    """Holiday dates from data/kb files. Printed at start-up so you can verify them."""
    hs = set(extra)
    def grab(txt):
        for m in DATE_ISO.finditer(txt):
            d = _mk_date(*m.groups()); d and hs.add(d)
        for m in DATE_DMY.finditer(txt):
            if m.group(3):
                d = _mk_date(m.group(3), MONTH_NUM[m.group(2)[:3].lower()], m.group(1)); d and hs.add(d)
    for p in list(DATA_DIR.glob("*holiday*")) + list(KB_DIR.glob("*holiday*")):
        grab(p.read_text(encoding="utf-8", errors="ignore"))
    for p in KB_DIR.glob("*.md"):
        for part in re.split(r"(?m)(?=^##\s*\[)", p.read_text(encoding="utf-8", errors="ignore")):
            if "holiday" in part.lower(): grab(part)
    return hs

def next_business_days(start, n, holidays=()):
    out, d = [], start
    while len(out) < n:
        d += timedelta(days=1)
        if d.weekday() != 6 and d not in holidays: out.append(d)
    return out

def date_helper_block(message, holidays):
    dates = extract_dates(message)
    if not dates: return ""
    lines = ["DATE HELPER (computed by code - use these dates, do not do your own date arithmetic):",
             f"- Today (reference date): {REFERENCE_DATE} ({REFERENCE_DATE:%A})"]
    for d in dates[:2]:
        bd = next_business_days(d, 15, holidays)
        lines.append(f"- Date mentioned: {d} ({d:%A})")
        lines.append("  Business days after it (Mon-Sat, excluding holidays; day 1 = first business day AFTER that date): " +
                     ", ".join(f"{i+1}={x:%Y-%m-%d}({x:%a})" for i, x in enumerate(bd)))
        lines.append("  Calendar days after it: " + ", ".join(f"+{k}={(d + timedelta(days=k)):%Y-%m-%d}" for k in (1, 2, 3, 5, 7, 10, 14, 15, 30)))
    if holidays: lines.append("- Holidays considered: " + ", ".join(sorted(str(h) for h in holidays if abs((h - REFERENCE_DATE).days) < 120)))
    lines.append("- If the policy counts days differently from this convention, follow the policy wording.")
    return "\n".join(lines)

_LANG_ROMAN_HI = re.compile(r"\b(hai|hain|kya|mujhe|mera|meri|mere|kab|kaise|kitne|kitna|nahi|karna|karo|kar|chahiye|wala|wali|aap|hoon|hun|sakta|sakti|kahan|dena|gaya|hua|ko|pe|mein|se|ka|ki|ke|aur|abhi|wapas|vapas|paise|din)\b", re.I)
_STRONG_HI = {"hai", "hain", "kya", "mujhe", "mera", "meri", "chahiye", "kahan", "kitne", "kaise", "nahi", "wapas", "vapas", "hoon"}
def detect_lang(text):
    if re.search(r"[\u0900-\u097F]", text): return "hi"
    hits = [h.lower() for h in _LANG_ROMAN_HI.findall(text)]
    if any(h in _STRONG_HI for h in hits) or len(hits) >= 3: return "hinglish"
    return "en"

In [7]:
TEXT_COLUMNS = {"order_id", "customer_id", "delivery_pincode", "return_rrn", "cancellation_rrn", "rto_rrn", "sku", "return_sku", "damage_sku", "pincode", "phone"}
def _clean_row(row):
    out = {}
    for k, v in row.items():
        if v == "": out[k] = None
        elif v in ("True", "False"): out[k] = (v == "True")
        elif k not in TEXT_COLUMNS and v.lstrip("-").isdigit(): out[k] = int(v)
        else: out[k] = v
    return out

def read_rows(name):
    path = DATA_DIR / name
    if not path.exists(): return []
    with path.open(encoding="utf-8", newline="") as f:
        return [_clean_row(r) for r in csv.DictReader(f)]

ORDERS = {r["order_id"]: r for r in read_rows("orders.csv")}
ITEMS = {}
for _it in read_rows("order_items.csv"): ITEMS.setdefault(_it["order_id"], []).append(_it)
CUSTOMERS = {r["customer_id"]: r for r in read_rows("customers.csv")}
ORDER_ID_LEN = Counter(len(k) for k in ORDERS).most_common(1)[0][0] if ORDERS else 8
HOLIDAYS = load_holidays()
print(f"Loaded orders={len(ORDERS)} customers={len(CUSTOMERS)} order-id length={ORDER_ID_LEN}")
print("Holidays detected (VERIFY against the KB calendar):", sorted(str(h) for h in HOLIDAYS) or "none found")

SECRET_FIELDS = {"delivery_pincode", "pincode", "phone", "customer_id"}
def public_view(obj):
    """Remove verification secrets / internal keys before anything reaches the model or a log."""
    if isinstance(obj, dict): return {k: public_view(v) for k, v in obj.items() if k not in SECRET_FIELDS | {"verified", "found"} and v is not None}
    if isinstance(obj, list): return [public_view(x) for x in obj]
    return obj

def lookup_order(order_id, verification_value=None):
    oid = "".join(c for c in normalize_digits(order_id) if c.isdigit())
    if verification_value is None or not str(verification_value).strip():
        return {"verification_required": True, "order_id": oid, "message": "Ask for registered phone last 4 digits OR delivery pincode."}
    proof = "".join(c for c in normalize_digits(verification_value) if c.isdigit())
    order = ORDERS.get(oid)
    if order:
        customer = CUSTOMERS.get(order["customer_id"])
        if customer:
            ok = (len(proof) == 4 and proof == str(customer["phone"])[-4:]) or (len(proof) == 6 and proof == str(order["delivery_pincode"]))
            if ok:
                return {"found": True, "verified": True, **order, "items": ITEMS.get(oid, []),
                        "customer_first_name": str(customer["name"]).split()[0], "is_plus": customer["is_plus"]}
    return {"verified": False, "verification_required": True, "message": "Verification failed or order not found."}

def create_ticket(summary, order_id=None, reason=None, intent=None, tickets_file=None):
    if not summary or len(summary.strip()) < 20: return {"created": False, "error": "summary too short"}
    tickets_file = Path(tickets_file or TICKETS_FILE)
    tid = "TKT" + uuid.uuid4().hex[:12].upper()
    row = {"ticket_id": tid, "created_at": datetime.now().isoformat(timespec="seconds"), "order_id": order_id or "", "intent": intent or "", "reason": reason or "", "summary": summary}
    fresh = not tickets_file.exists()
    with tickets_file.open("a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row))
        if fresh: w.writeheader()
        w.writerow(row)
    return {"created": True, "ticket_id": tid}

SKU_RE = re.compile(r"\b[A-Z]{3}-\d{4}\b", re.I)
def _norm_name(s): return re.sub(r"[^a-z0-9 ]+", " ", str(s).lower()).split()

class ProductCatalog:
    """Structured SKU / product-name lookup so product facts never depend on embeddings."""
    def __init__(self, rows):
        self.rows = rows
        self.name_col = next((c for c in ("product_name", "name", "title") if rows and c in rows[0]), None)
    def find(self, text, limit=3):
        if not self.rows: return []
        hits = []
        for s in SKU_RE.findall(text):
            hits += [r for r in self.rows if str(r.get("sku", "")).upper() == s.upper()]
        if hits: return hits[:limit]
        if not self.name_col: return []
        t = set(_norm_name(text)); tl = " ".join(_norm_name(text)); scored = []
        for r in self.rows:
            toks = _norm_name(r[self.name_col])
            if len(toks) < 2: continue
            phrase = " ".join(toks)
            if phrase in tl: score = 1.0 + len(toks) / 100
            else:
                overlap = sum(1 for x in toks if x in t) / len(toks)
                score = overlap if overlap >= 0.85 else 0
            if score: scored.append((score, r))
        scored.sort(key=lambda x: -x[0])
        if not scored: return []
        best = scored[0][0]
        return [r for s, r in scored if s >= best - 1e-9][:limit]
    @staticmethod
    def retrieval_terms(rows):
        terms = []
        for r in rows:
            for k, v in r.items():
                if v is not None and re.search(r"categ|class|type|return|warrant|assured", k, re.I): terms.append(str(v))
        return " ".join(terms)

CATALOG = ProductCatalog(read_rows("products.csv"))
print("Product rows:", len(CATALOG.rows), "name column:", CATALOG.name_col)

Loaded orders=1000 customers=300 order-id length=8
Holidays detected (VERIFY against the KB calendar): ['2026-08-15', '2026-10-02', '2026-10-19', '2026-11-09']
Product rows: 70 name column: title


In [8]:
def parse_date_str(s):
    m = DATE_ISO.search(s or "")
    return _mk_date(*m.groups()) if m else None

def load_docs(kb_dir=None):
    kb_dir = Path(kb_dir or KB_DIR); docs = []
    for path in sorted(kb_dir.glob("*.md")):
        text = path.read_text(encoding="utf-8"); meta = {}
        for key in ("doc_id", "title", "doc_type", "effective_from", "status", "superseded_by"):
            m = re.search(rf"(?m)^{key}:\s*(.+)$", text)
            if m: meta[key] = m.group(1).strip().strip('"')
        for part in re.split(r"(?m)(?=^##\s*\[[^\]]+\])", text):
            if not part.strip(): continue
            m = re.search(r"(?m)^##\s*\[([^\]]+)\]\s*(.*)$", part)
            if not m:
                if path.name.startswith("00_about"): docs.append({"section_id": "KB_INTRO", "file": path.name, "metadata": meta, "content": part})
                continue
            docs.append({"section_id": m.group(1).strip(), "file": path.name, "metadata": meta, "content": part.strip()})
    return docs

def is_active(doc):
    """Excludes superseded docs and docs not yet effective on the reference date."""
    meta, fname = doc["metadata"], doc["file"].lower()
    if meta.get("status", "").lower() == "superseded" or "superseded" in fname: return False
    if meta.get("superseded_by", "").strip().lower() not in ("", "none", "null", "-"): return False
    eff = parse_date_str(meta.get("effective_from"))
    if eff and eff > REFERENCE_DATE: return False
    return True

def authority_weight(doc):
    """Explicit source precedence: policy > agent guide > help/FAQ > marketing > logs (as stated in the system prompt)."""
    meta = doc["metadata"]
    for field in ((meta.get("doc_type") or ""), meta.get("title", "") + " " + doc["file"]):
        t = field.lower()
        if not t.strip(): continue
        if re.search(r"\b(log|logs|historical|archive|archived|chat)\b", t): return 0.40
        if re.search(r"marketing|promo|campaign|blog", t): return 0.55
        if re.search(r"faq|help|support[_ ]?center", t): return 0.70
        if re.search(r"agent|guide|playbook|handbook|internal", t): return 0.80
        if re.search(r"polic|terms|rules|standard|about", t): return 1.00
    return 0.75

QUERY_EXPANSIONS = [  # cross-lingual hints for the RETRIEVAL query only (never shown to the customer)
    (r"वापस|लौटा|wapas|vapas|\breturn", "return policy return window"),
    (r"रिफ़ंड|रिफंड|रिफ|पैसे|paise|refund", "refund timeline refund method"),
    (r"कैंसिल|cancel", "cancel order cancellation"),
    (r"शिप|\bship", "shipped dispatch"),
    (r"टूट|क्रैक|toota|damage|broken|crack|torn|faulty", "damaged item damage on arrival"),
    (r"वारंटी|warranty|guarantee|assured", "warranty brand warranty ZentroMart Assured claim"),
    (r"डिलीवर|deliver|पहुँच", "delivery"),
    (r"कूपन|coupon", "coupon"),
    (r"इंसान|insaan|human|person|agent", "human agent escalation"),
    (r"\bupi\b", "UPI refund"),
    (r"नेट\s*बैंकिंग|net\s*banking|netbanking", "net banking refund"),
    (r"क्रेडिट\s*कार्ड|credit\s*card", "credit card refund"),
    (r"\bcod\b|cash on delivery|कैश", "cash on delivery refund"),
    (r"दुकान|store|shop", "store offline pickup"),
]
def expand_query(text):
    extra = [e for pat, e in QUERY_EXPANSIONS if re.search(pat, text, re.I)]
    return (text + " " + " ".join(extra)).strip()

class Retriever:
    """Hybrid (multilingual dense + TF-IDF) retrieval fused with RRF, then re-weighted by source authority."""
    def __init__(self, kb_dir=None, use_dense=True, model_name=EMBED_MODEL, chunk_words=90):
        alldocs = load_docs(kb_dir)
        self.docs = [d for d in alldocs if is_active(d)]
        self.excluded = [d["section_id"] + "@" + d["file"] for d in alldocs if not is_active(d)]
        if not self.docs: raise ValueError("No active KB sections found")
        self.weights = [authority_weight(d) for d in self.docs]
        self.section_ids = {d["section_id"] for d in self.docs}
        from sklearn.feature_extraction.text import TfidfVectorizer
        texts = [(d["metadata"].get("title", "") + " " + d["content"]) for d in self.docs]
        self.tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, token_pattern=r"(?u)\b\w+\b")
        self.tmat = self.tfidf.fit_transform(texts)
        self.dense = None
        if use_dense:
            try:
                from sentence_transformers import SentenceTransformer
                self.model = SentenceTransformer(model_name)
                chunks, owner = [], []
                for i, d in enumerate(self.docs):
                    words = d["content"].split(); head = d["metadata"].get("title", "") + " | " + d["content"].splitlines()[0]
                    step = max(1, chunk_words - 20)
                    for s in range(0, max(1, len(words)), step):
                        chunks.append(head + ": " + " ".join(words[s:s + chunk_words])); owner.append(i)
                        if s + chunk_words >= len(words): break
                self.owner = np.array(owner)
                self.dense = self.model.encode(chunks, normalize_embeddings=True, show_progress_bar=False).astype("float32")
            except Exception as e:
                print("WARNING: dense retrieval unavailable, using lexical only:", e)
    def search(self, query, k=6, pool=20):
        from sklearn.metrics.pairwise import linear_kernel
        q = expand_query(query); n = len(self.docs); fused = np.zeros(n)
        lex = linear_kernel(self.tfidf.transform([q]), self.tmat)[0]
        for r, i in enumerate(np.argsort(-lex)[:pool]):
            if lex[i] > 0: fused[i] += 1.0 / (60 + r)
        if self.dense is not None:
            qe = self.model.encode([q], normalize_embeddings=True).astype("float32")[0]
            sims = self.dense @ qe; best = np.full(n, -1.0)
            for ci, o in enumerate(self.owner): best[o] = max(best[o], sims[ci])
            for r, i in enumerate(np.argsort(-best)[:pool]): fused[i] += 1.0 / (60 + r)
        final = fused * (0.5 + 0.5 * np.array(self.weights))
        order = [i for i in np.argsort(-final)[:k] if final[i] > 0]
        return [{**self.docs[i], "score": float(final[i]), "authority": self.weights[i]} for i in order]

_RETRIEVER = None
def get_retriever():
    global _RETRIEVER
    if _RETRIEVER is None: _RETRIEVER = Retriever()
    return _RETRIEVER

def kb_audit(retr=None):
    """Print what was indexed so section IDs / authority weights can be sanity-checked."""
    retr = retr or get_retriever()
    rows = pd.DataFrame([{"file": d["file"], "doc_type": d["metadata"].get("doc_type"), "status": d["metadata"].get("status"),
                          "effective_from": d["metadata"].get("effective_from"), "authority": w} for d, w in zip(retr.docs, retr.weights)]).drop_duplicates()
    print("Active KB files and authority weights:"); display(rows)
    print("Excluded (superseded / not yet effective):", retr.excluded)
    dup = [k for k, v in Counter(d["section_id"] for d in retr.docs).items() if v > 1]
    print("Duplicate section IDs across active docs:", dup or "none")
    return rows

def gold_coverage_check(retr=None):
    """Which gold section IDs in practice/questions.csv are NOT in the active index? (retrieval can never find them)"""
    retr = retr or get_retriever()
    q = pd.read_csv(PRACTICE_DIR / "questions.csv", dtype=str, keep_default_na=False)
    gold = {g.strip() for cell in q.get("gold_evidence", []) for g in cell.split(";") if g.strip() and not g.strip().startswith("products.csv")}
    missing = sorted(gold - retr.section_ids)
    print("Gold section IDs missing from active index:", missing or "none")
    return missing

In [9]:
T = {
 "greet": {"en": "Hi, I'm ZentroMart's AI assistant. How can I help you today?",
           "hi": "नमस्ते, मैं ZentroMart का AI असिस्टेंट हूँ। मैं आपकी कैसे मदद कर सकता हूँ?",
           "hinglish": "Namaste, main ZentroMart ka AI assistant hoon. Main aapki kaise madad kar sakta hoon?"},
 "ask_order": {"en": "Happy to help. Could you please share your order number?",
               "hi": "ज़रूर, मैं मदद करूँगा। कृपया अपना ऑर्डर नंबर बताइए।",
               "hinglish": "Zaroor, main madad karunga. Kripya apna order number bataiye."},
 "ask_verify": {"en": "Thank you. To protect your account, please share either the last 4 digits of the phone number registered on this order, or its delivery pincode.",
                "hi": "धन्यवाद। आपकी सुरक्षा के लिए, कृपया इस ऑर्डर से जुड़े रजिस्टर्ड फ़ोन नंबर के आख़िरी 4 अंक या डिलीवरी पिनकोड बताइए।",
                "hinglish": "Dhanyavaad. Aapki suraksha ke liye, kripya is order ke registered phone number ke aakhri 4 digits ya delivery pincode bataiye."},
 "verify_retry": {"en": "I couldn't verify those details. Please check and share either the last 4 digits of the registered phone number or the delivery pincode once more.",
                  "hi": "मैं इन जानकारियों की पुष्टि नहीं कर पाया। कृपया रजिस्टर्ड फ़ोन नंबर के आख़िरी 4 अंक या डिलीवरी पिनकोड एक बार फिर जाँचकर बताइए।",
                  "hinglish": "Main in details ko verify nahi kar paya. Kripya registered phone ke aakhri 4 digits ya delivery pincode ek baar phir check karke bataiye."},
 "verify_locked": {"en": "I'm unable to verify this order, so I can't share its details. I can still help with general policy questions, or connect you with a human colleague.",
                   "hi": "मैं इस ऑर्डर की पुष्टि नहीं कर पा रहा हूँ, इसलिए इसकी जानकारी साझा नहीं कर सकता। मैं सामान्य पॉलिसी के सवालों में मदद कर सकता हूँ, या आपको किसी मानव सहयोगी से जोड़ सकता हूँ।",
                   "hinglish": "Main is order ko verify nahi kar pa raha hoon, isliye iski details share nahi kar sakta. Main general policy sawaalon mein madad kar sakta hoon, ya aapko human colleague se jod sakta hoon."},
 "verified": {"en": "Thank you, you're verified. Order {order_id}: {summary}. What would you like to know about it?",
              "hi": "धन्यवाद, आपकी पुष्टि हो गई है। ऑर्डर {order_id}: {summary}। आप इसके बारे में क्या जानना चाहेंगे?",
              "hinglish": "Dhanyavaad, aapka verification ho gaya hai. Order {order_id}: {summary}. Aap iske baare mein kya jaanna chahenge?"},
 "ticket_ok": {"en": "I've created a support ticket ({tid}) so a human colleague can review this and follow up with you.",
               "hi": "मैंने एक सपोर्ट टिकट ({tid}) बना दिया है ताकि कोई मानव सहयोगी इसे देखकर आपसे संपर्क कर सके।",
               "hinglish": "Maine ek support ticket ({tid}) bana diya hai taaki koi human colleague ise dekhkar aapse sampark kar sake."},
 "ticket_fail": {"en": "I couldn't confirm that a ticket was created. Please contact ZentroMart support directly so a human colleague can help.",
                 "hi": "मैं पुष्टि नहीं कर पाया कि टिकट बना है। कृपया सीधे ZentroMart सपोर्ट से संपर्क करें।",
                 "hinglish": "Main confirm nahi kar paya ki ticket bana hai. Kripya seedhe ZentroMart support se sampark karein."},
 "fallback": {"en": "I'm sorry, I couldn't find a reliable answer to that. I can connect you with a human colleague if you'd like.",
              "hi": "क्षमा करें, मुझे इसका भरोसेमंद उत्तर नहीं मिल पाया। अगर आप चाहें तो मैं आपको किसी मानव सहयोगी से जोड़ सकता हूँ।",
              "hinglish": "Maaf kijiye, mujhe iska bharosemand jawaab nahi mila. Aap chahein to main aapko human colleague se jod sakta hoon."},
}
INTRO = {"en": "Hi, I'm ZentroMart's AI assistant. ", "hi": "नमस्ते, मैं ZentroMart का AI असिस्टेंट हूँ। ", "hinglish": "Namaste, main ZentroMart ka AI assistant hoon. "}
LANG_NAME = {"en": "English", "hi": "Hindi (Devanagari script)", "hinglish": "Hinglish (Hindi written in Roman letters)"}

HUMAN_WORD = re.compile(r"\b(human|real person|a person|someone|representative|executive|manager|supervisor|live agent|customer care|agent)\b|इंसान|इन्सान|मानव|मैनेजर|एजेंट|\binsaan\b|\baadmi\b", re.I)
HUMAN_VERB = re.compile(r"speak|talk|baat|connect|transfer|chat|call|बात|मिला|जोड़|want|need|chahiye|चाहिए", re.I)
REPEAT_CONTACT = re.compile(r"(third|3rd|fourth|4th|teesri|teesra|तीसरी|तीसरा|चौथी)\s+(time|baar|बार)|(called|calling|contacted|emailed|complained)\D{0,25}(three|3|multiple|several|many)\s+times|baar\s*baar|कई\s*बार|बार-बार|तीसरी\s*बार", re.I)
DAMAGE_WORD = re.compile(r"damage|damaged|broken|crack|cracked|defective|toota|tuta|टूट|क्रैक|खराब", re.I)
APPROVE_WORD = re.compile(r"\bapprove|\bauthori[sz]e|override|approval|अप्रूव|मंज़ूर|manzoor", re.I)
ORDER_WORD = re.compile(r"\border\b|ऑर्डर", re.I)
HELP_WORD = re.compile(r"\bhelp\b|madad|मदद|sahayata|सहायता", re.I)
GREETING_ONLY = re.compile(r"^\W*(hi|hello|hey|namaste|namaskar|नमस्ते|हेलो|हैलो)\W*$", re.I)
ORDER_DATA_NEEDED = [re.compile(p, re.I) for p in [
    r"where\s+is\s+(my\s+)?(order|package|parcel)", r"(order|package|parcel)\W+(\w+\W+){0,6}(hasn'?t|has\s+not|not\s+yet)\s+(arrived|come|reached|been\s+delivered)",
    r"(order|ऑर्डर)\W+(\w+\W+){0,4}(kahan|kahaan|कहा)", r"(refund|रिफ)\W+(\w+\W+){0,4}(kahan|kahaan|कहा|where|status)", r"where\W+(\w+\W+){0,3}refund",
    r"\bcancel\s+(it|this)\b|\bcancel\s+(kar\s*do|karna|karo)\b|\b(ise|isko)\s+cancel|इसे\s+कैंसिल|इसको\s+कैंसिल", r"when\s+(was|did)\s+(it|my\s+order)\s+ship|kab\s+ship|कब\s+शिप",
    r"track(ing)?\s+(my\s+)?(order|package|parcel)", r"(order|ऑर्डर)\s+status"]]

def escalation_reason(text):
    if HUMAN_WORD.search(text) and HUMAN_VERB.search(text): return "customer asked for a human agent"
    if REPEAT_CONTACT.search(text): return "repeat contact about the same issue"
    if DAMAGE_WORD.search(text) and APPROVE_WORD.search(text): return "damage report needing human approval of a replacement/refund"
    return None

GENERIC_GREETING = re.compile(r"how can i (help|assist)( you)?( today)?|what can i help you with|kaise madad|कैसे मदद", re.I)
STALL = re.compile(r"\bplease\s+(wait|hold)\b|\b(one|a\s+(?:little\s+)?)?(moment|second)\s+please|\bjust\s+a\s+moment\b|\bhold\s+on\b|\bwhile\s+i\s+(check|verify|look)|\bi(?:'m|\s+am)\s+(?:currently\s+)?(?:checking|verifying|looking\s+(?:up|into)|fetching|retrieving)\b|\blet\s+me\s+(check|verify|look)", re.I)
PLACEHOLDER = re.compile(r"\[[^\]\n]{2,40}\]|\{[^}\n]{2,40}\}")
TICKET_CLAIM = re.compile(r"\b(ticket|case)\b[^.\n]{0,60}\b(created|raised|logged|registered|opened|generated|submitted)\b|\b(created|raised|logged|opened|generated|submitted)\b[^.\n]{0,40}\b(ticket|case)\b|\bTKT[0-9A-F]{6,}\b", re.I)

def is_generic_greeting(answer):
    words = re.findall(r"\w+", answer)
    return len(words) <= 32 and bool(GENERIC_GREETING.search(answer))

def strip_internal_ids(answer, ids):
    if ids:
        pat = "|".join(re.escape(i) for i in sorted(ids, key=len, reverse=True))
        answer = re.sub(rf"(?i)\b(?:according\s+to|based\s+on|as\s+per|per)\s+(?:the\s+)?(?:section\s+)?[\[\(]?(?:{pat})[\]\)]?\s*[,:]?\s*", "", answer)
        answer = re.sub(rf"\s*[\[\(]\s*(?:{pat})(?:\s*[,;]\s*(?:{pat}))*\s*[\]\)]", "", answer)
        answer = re.sub(rf"\b(?:{pat})\b", "", answer)
    answer = re.sub(r"[ \t]{2,}", " ", answer); answer = re.sub(r"\s+([,.;:!?])", r"\1", answer).strip()
    return answer[:1].upper() + answer[1:] if answer else answer

def strip_ticket_claims(answer):
    parts = re.split(r"(?<=[.!?])\s+|\n+", answer)
    return " ".join(p for p in parts if p and not TICKET_CLAIM.search(p)).strip()

def answer_problem(answer, message):
    if not answer.strip(): return "empty"
    if not GREETING_ONLY.match(message.strip()) and is_generic_greeting(answer): return "generic_greeting"
    if STALL.search(answer): return "stall"
    if PLACEHOLDER.search(answer): return "placeholder"
    return None

In [10]:
def rx(p, f=re.I): return re.compile(unicodedata.normalize("NFC", p), f)
def nfc(s): return unicodedata.normalize("NFC", s)

CUE_PIN = rx(r"pin|zip|पिन")
CUE_PHONE = rx(r"phone|mobile|last|digit|number|four|फ़ोन|फोन|अंक|नंबर|आख़िरी|आखिरी|चार|aakhri|akhri")
FILLER = {nfc(w) for w in """last four digits digit hain hai is are the my phone number pincode pin code ok okay theek thanks thank you of for it
 registered mobile delivery hi hello pe ka ke ki yes haan han 4 6 हैं है आख़िरी आखिरी चार अंक पिन कोड फ़ोन फोन नंबर के का की हूँ जी हाँ""".split()}
MAX_ATTEMPTS_PER_ORDER, MAX_TOTAL_FAILURES = 2, 4

def _near_month(text, span):
    w = text[max(0, span[0] - 10): span[1] + 10].lower()
    return bool(re.search(rf"\b{MONTH_RE}\b", w))

def parse_number_groups(text):
    """Standalone digit groups ("8978 2837" is one group). Never concatenates unrelated digits."""
    out = []
    for m in GROUP_RE.finditer(text):
        d = re.sub(r"\D", "", m.group()); isd = _near_month(text, m.span())
        if not isd and len(d) in (ORDER_ID_LEN + 4, ORDER_ID_LEN + 6):      # order id glued to a code
            out += [{"digits": d[:ORDER_ID_LEN], "span": m.span(), "is_date": False}, {"digits": d[ORDER_ID_LEN:], "span": m.span(), "is_date": False}]
        else:
            out.append({"digits": d, "span": m.span(), "is_date": isd})
    return out

def order_summary(view):
    status = next((view[k] for k in ("order_status", "status", "current_status") if view.get(k)), None)
    names = []
    for it in view.get("items", []):
        n = next((it[k] for k in ("product_name", "name", "title", "sku") if it.get(k)), None)
        if n: names.append(str(n))
    parts = ([f"status {status}"] if status else []) + ([f"items: {', '.join(names[:3])}"] if names else [])
    return "; ".join(parts) or "I've found it"

class ZentroMartAgentLocal:
    def __init__(self, use_rag=True, generator=None, retriever=None, catalog=None, holidays=None, tickets_file=None):
        self.use_rag = use_rag
        self.generator = generator or globals().get("local_generator")
        self.retriever = (retriever or get_retriever()) if use_rag else None
        self.catalog = catalog or CATALOG
        self.holidays = HOLIDAYS if holidays is None else holidays
        self.tickets_file = tickets_file
        self.reset()
    def reset(self):
        self.history = []; self.pending_order_id = None; self.verified_order = None; self.verification_attempts = 0
        self.locked_orders = set(); self.total_failures = 0; self.secrets = set(); self.ticket_ids = []
        self.last_sources = []; self.last_product = []; self.active_products = []; self.turn_tools = []; self.last_guard = ""
    @property
    def last_tools(self): return "; ".join(self.turn_tools)
    def _mask(self, text):
        for s in self.secrets: text = text.replace(s, "****")
        return text

    # ---- verification state machine -------------------------------------------------
    def _update_order_state(self, norm):
        items = parse_number_groups(norm); ev = None
        cands = [i for i in items if not i["is_date"] and (i["digits"] in ORDERS or len(i["digits"]) == ORDER_ID_LEN)]
        if cands:
            oid = cands[0]["digits"]
            if oid != self.pending_order_id:
                self.pending_order_id, self.verified_order, self.verification_attempts = oid, None, 0
                lookup_order(oid); self.turn_tools.append(f"lookup_order(order_id={oid}) -> verification_required")
                ev = "order_given"
        if not self.pending_order_id or self.verified_order is not None: return ev
        vc = [i for i in items if not i["is_date"] and len(i["digits"]) in (4, 6) and i["digits"] != self.pending_order_id
              and not any(c["span"] == i["span"] for c in cands)]
        chosen = None
        v4 = [c for c in vc if len(c["digits"]) == 4]; v6 = [c for c in vc if len(c["digits"]) == 6]
        if CUE_PIN.search(norm) and v6: chosen = v6[0]
        elif CUE_PHONE.search(norm) and v4: chosen = v4[0]
        elif len(vc) == 1 and (len(norm.split()) <= 6 or CUE_PIN.search(norm) or CUE_PHONE.search(norm)): chosen = vc[0]
        if chosen is None: return ev
        if self.pending_order_id in self.locked_orders or self.total_failures >= MAX_TOTAL_FAILURES: return "locked"
        self.secrets.add(chosen["digits"]); self.verification_attempts += 1
        res = lookup_order(self.pending_order_id, chosen["digits"])
        self.turn_tools.append(f"lookup_order(order_id={self.pending_order_id}, verification_value=****) -> verified={bool(res.get('verified'))}")
        if res.get("verified"):
            self.verified_order = public_view(res); return "verified"
        self.total_failures += 1
        if self.verification_attempts >= MAX_ATTEMPTS_PER_ORDER:
            self.locked_orders.add(self.pending_order_id); return "locked"
        return "failed"

    @staticmethod
    def _is_pure_verification(norm):
        toks = [t for t in re.findall(r"[^\s।,.!?;:]+", nfc(re.sub(r"\d+", " ", norm.lower()))) if t not in FILLER]
        return len(toks) <= 1

    # ---- deterministic routes ---------------------------------------------------------
    def _route(self, norm, lang):
        ev = self._update_order_state(norm)
        reason = escalation_reason(norm)
        if reason: return self._escalate(norm, lang, reason)
        if ev == "locked": return T["verify_locked"][lang]
        if ev == "failed": return T["verify_retry"][lang]
        if ev == "order_given": return T["ask_verify"][lang]
        if ev == "verified" and self._is_pure_verification(norm):
            return T["verified"][lang].format(order_id=self.pending_order_id, summary=order_summary(self.verified_order))
        if GREETING_ONLY.match(norm.strip()): return T["greet"][lang]
        if self.verified_order is None:
            needs = any(p.search(norm) for p in ORDER_DATA_NEEDED)
            order_help = bool(ORDER_WORD.search(norm) and HELP_WORD.search(norm) and len(norm.split()) <= 14)
            if needs or order_help:
                if not self.pending_order_id: return T["ask_order"][lang]
                if self.pending_order_id in self.locked_orders: return T["verify_locked"][lang]
                return T["ask_verify"][lang]
        return None

    def _escalate(self, norm, lang, reason):
        prev = [m["content"] for m in self.history if m["role"] == "user"][-2:]
        oid = self.pending_order_id
        status = "verified" if self.verified_order else "NOT verified"
        summary = (f"Reason: {reason}. Customer said: {self._mask(norm)}. Earlier customer messages: {' | '.join(prev) or 'none'}. "
                   f"Order: {oid or 'not provided'} ({status}). Reference date {REFERENCE_DATE}.")[:900]
        res = create_ticket(summary, order_id=oid, reason=reason, intent="escalate", tickets_file=self.tickets_file)
        self.turn_tools.append(f"create_ticket() -> created={res.get('created')}" + (f" ticket_id={res['ticket_id']}" if res.get("created") else ""))
        if res.get("created") and res.get("ticket_id"):
            self.ticket_ids.append(res["ticket_id"]); return T["ticket_ok"][lang].format(tid=res["ticket_id"])
        return T["ticket_fail"][lang]

    # ---- LLM path -----------------------------------------------------------------------
    def _generate(self, messages):
        prompt = self.generator.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        out = self.generator(prompt, max_new_tokens=320, do_sample=False, return_full_text=False)
        return out[0]["generated_text"].strip()

    def _retrieval_query(self, norm, products):
        prev = [m["content"] for m in self.history if m["role"] == "user" and len(m["content"].split()) > 3 and not re.fullmatch(r"[\W\d]+", m["content"])]
        q = (prev[-1] + " " + norm) if (len(norm.split()) < 10 and prev) else norm
        if products: q += " " + ProductCatalog.retrieval_terms(products)
        return q

    def _build_user_prompt(self, norm, lang, evidence, products):
        parts = [SYSTEM_PROMPT, "",
                 "RULES FOR THIS REPLY:",
                 "- Answer the customer's question directly now. Do NOT reply with only a greeting and do not introduce yourself (the system adds the introduction).",
                 "- Use ONLY the information below. If it does not answer the question, say so briefly and offer a human colleague. Never invent policy.",
                 "- If sources conflict, follow the one with the highest authority and the latest effective date.",
                 "- Never mention source IDs, file names, 'evidence', 'FAQ' or internal tags to the customer.",
                 "- Never say you are 'checking', 'verifying' or 'looking up' anything: every lookup is already done and shown below. Never leave placeholders like [Date].",
                 "- Do not ask for information the customer already gave, or that appears below. Ask at most one question, only if a required detail is truly missing.",
                 "- If the policy does not allow what is asked (for example cancelling an order that already shipped, or vouchers/compensation that the policy does not offer), say so politely in one sentence and give the allowed alternatives. Never promise anything the policy does not state.",
                 f"- Reply in {LANG_NAME[lang]}.", ""]
        if evidence: parts += ["POLICY EVIDENCE (highest authority first):", evidence, ""]
        if products: parts += ["PRODUCT CATALOG RESULT:", json.dumps([{k: v for k, v in p.items() if v is not None} for p in products], ensure_ascii=False, default=str), ""]
        if self.verified_order: parts += ["VERIFIED ORDER DATA (the caller passed verification; share only what is relevant):", json.dumps(public_view(self.verified_order), ensure_ascii=False, default=str), ""]
        elif self.pending_order_id: parts += ["ORDER STATUS: caller NOT verified. Do not reveal any order-specific information.", ""]
        dh = date_helper_block(norm, self.holidays)
        if dh: parts += [dh, ""]
        parts += ["CUSTOMER MESSAGE:", norm]
        return "\n".join(parts)

    def _clean(self, raw):
        ids = self.retriever.section_ids if self.retriever else set()
        return strip_ticket_claims(strip_internal_ids(raw, ids))

    def _llm_answer(self, norm, lang):
        hist = self.history[-6:]
        if not self.use_rag:                                               # prompt-only baseline
            msgs = hist + [{"role": "user", "content": SYSTEM_PROMPT + "\n\nCustomer Query: " + norm}]
            return self._generate(msgs) or T["fallback"][lang]
        products = self.catalog.find(norm)
        if products: self.active_products = products
        products = products or self.active_products
        self.last_product = products
        hits = self.retriever.search(self._retrieval_query(norm, products), k=6)
        self.last_sources = [{"section_id": d["section_id"], "file": d["file"], "score": d["score"], "text": d["content"]} for d in hits]
        ordered = sorted(hits, key=lambda d: (-d["authority"], -d["score"]))
        evidence = "\n\n".join(f"[Source {i+1} | ID {d['section_id']} | authority={d['authority']:.2f} | status={d['metadata'].get('status','unknown')} | effective_from={d['metadata'].get('effective_from','unknown')}]\n{d['content'][:1800]}" for i, d in enumerate(ordered))
        user_prompt = self._build_user_prompt(norm, lang, evidence, products)
        answer = self._clean(self._generate(hist + [{"role": "user", "content": user_prompt}]))
        problem = answer_problem(answer, norm); self.last_guard = problem or ""
        if problem:
            retry = user_prompt + f"\n\nIMPORTANT: your previous draft was rejected ({problem}). Write the final customer reply now, using the information above, with no greeting, no 'checking' or 'please wait', and no placeholders."
            answer = self._clean(self._generate(hist + [{"role": "user", "content": retry}]))
            if answer_problem(answer, norm): answer = T["fallback"][lang]; self.last_guard += "+fallback"
        return answer

    def respond(self, message, conversation_id="default"):
        self.turn_tools = []; self.last_sources = []; self.last_product = []; self.last_guard = ""
        norm = nfc(normalize_digits(message)); lang = detect_lang(norm)
        answer = self._route(norm, lang) if self.use_rag else None
        if answer is None: answer = self._llm_answer(norm, lang)
        if not self.history and not re.search(r"AI (assistant|असिस्टेंट)", answer, re.I): answer = INTRO[lang] + answer
        self.history += [{"role": "user", "content": self._mask(norm)}, {"role": "assistant", "content": answer}]
        return answer

In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.getenv("HF_TOKEN")
print("CUDA available:", torch.cuda.is_available())
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=hf_token, device_map="auto",
                                             torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
local_generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("Loaded model:", MODEL_ID)

CUDA available: True


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Loaded model: google/gemma-2-2b-it


In [12]:
retr = get_retriever()
kb_audit(retr)
gold_coverage_check(retr)
print(ZentroMartAgentLocal(use_rag=True).respond("What is your return policy?"))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Active KB files and authority weights:


,file,doc_type,status,effective_from,authority
0,00_about_this_knowledge_base.md,meta,current,2026-08-01,1.00
5,agent_conversation_style.md,agent_guide,current,2026-07-01,0.80
10,agent_escalation_handoff.md,agent_guide,current,2026-07-01,0.80
14,agent_identity_verification.md,agent_guide,current,2026-07-01,0.80
18,faq_general.md,faq,current,2025-11-01,0.70
23,faq_hindi.md,faq,current,2026-08-01,0.70
27,help_reverse_pickup.md,help,current,2026-08-01,0.70
30,help_track_order.md,help,current,2026-02-01,0.70
32,help_troubleshooting_electronics.md,help,current,2026-05-01,0.70
36,mkt_plus_benefits_page.md,marketing,current,2026-01-01,0.55


Excluded (superseded / not yet effective): ['RETv3-2@pol_returns_v3_superseded.md', 'RETv3-6@pol_returns_v3_superseded.md']
Duplicate section IDs across active docs: none
Gold section IDs missing from active index: ['RETv3-2']
Hi, I'm ZentroMart's AI assistant. ZentroMart's return policy covers clothing and footwear within a 15-day return window.


In [13]:
def _gold_sections(cell):
    return [g.strip() for g in str(cell or "").split(";") if g.strip() and not g.strip().startswith("products.csv")]

def evaluate_single_turn(use_rag=True, limit=None):
    df = pd.read_csv(PRACTICE_DIR / "questions.csv", dtype=str, keep_default_na=False)
    if limit is not None: df = df.head(limit)
    agent = ZentroMartAgentLocal(use_rag=use_rag); rows = []
    for _, r in df.iterrows():
        agent.reset(); q = r.get("question", ""); answer = agent.respond(q); low = answer.lower()
        must = [x.strip() for x in r.get("must_include", "").split(" ; ") if x.strip()]
        bad = [x.strip() for x in r.get("must_not_include", "").split(" ; ") if x.strip()]
        include_ok = all(any(alt.strip().lower() in low for alt in item.split(" / ")) for item in must)
        no_bad = not any(x.lower() in low for x in bad)
        retrieved = [x["section_id"] for x in agent.last_sources]; gold = _gold_sections(r.get("gold_evidence", ""))
        rows.append({"id": r.get("id"), "question": q, "answer": answer, "expected_decision": r.get("expected_decision"),
                     "gold_evidence": r.get("gold_evidence"), "retrieved_sections": ";".join(retrieved),
                     "must_include_pass": include_ok, "must_not_include_pass": no_bad, "simple_pass": bool(include_ok and no_bad),
                     "gold_retrieved": (any(g in retrieved for g in gold) if gold else ""),
                     "product_gold": "products.csv" in str(r.get("gold_evidence", "")), "product_lookup_hit": bool(agent.last_product),
                     "generic_greeting": is_generic_greeting(answer), "guard_triggered": agent.last_guard, "tool_calls_observed": agent.last_tools})
    return pd.DataFrame(rows)

def summarize_csv(path):
    df = pd.read_csv(path, dtype=str, keep_default_na=False); b = lambda c: df[c].str.lower() == "true"
    n = len(df)
    return {"n": n, "simple_pass": int(b("simple_pass").sum()), "simple_pass_rate": float(b("simple_pass").mean()) if n else 0.0,
            "must_include_pass": int(b("must_include_pass").sum()), "must_not_include_pass": int(b("must_not_include_pass").sum()),
            "metric": "phrase-based (must_include / must_not_include substring checks); not a human judgement of correctness"}

def save_evaluation(df, name):
    path = OUTPUT_DIR / f"{name}.csv"; df.to_csv(path, index=False)
    summary = summarize_csv(path)                      # summary is always computed FROM the saved CSV
    (OUTPUT_DIR / f"{name}_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(name, summary); return summary

def archive_original_run():
    """Step 5.12: keep the original baseline. Put the OLD result files in OUTPUT_DIR before the first rerun."""
    arch = OUTPUT_DIR / "archive_original_run"
    if arch.exists(): print("Original run already archived at", arch); return
    old = [p for p in OUTPUT_DIR.glob("*") if p.is_file() and p.suffix in (".csv", ".json")]
    if not old: print("No previous result files found to archive (upload the old ones into", OUTPUT_DIR, "first if you want to keep them)."); return
    arch.mkdir(); [shutil.copy2(p, arch / p.name) for p in old]; print("Archived", len(old), "original files to", arch)

def build_failed_and_comparison():
    """Always regenerated from the CSVs currently on disk (never from stale objects)."""
    base = pd.read_csv(OUTPUT_DIR / "step1_prompt_only_final.csv", dtype=str, keep_default_na=False)
    rag = pd.read_csv(OUTPUT_DIR / "step2_rag_final.csv", dtype=str, keep_default_na=False)
    assert len(base) == 52 and len(rag) == 52, f"expected 52 rows, got {len(base)} / {len(rag)}"
    assert base["id"].tolist() == rag["id"].tolist(), "question IDs differ between runs"
    ok = lambda s: s.str.lower() == "true"
    failed = rag.loc[~ok(rag["simple_pass"])].copy(); failed.to_csv(OUTPUT_DIR / "step2_failed_cases_final.csv", index=False)
    comp = base[["id", "question", "answer", "simple_pass"]].merge(rag[["id", "answer", "simple_pass"]], on="id", suffixes=("_prompt_only", "_rag"), validate="one_to_one")
    p, r = ok(comp["simple_pass_prompt_only"]), ok(comp["simple_pass_rag"])
    comp["outcome"] = np.select([~p & r, p & ~r, p & r], ["Improved", "Regressed", "Passed both"], "Failed both")
    comp.to_csv(OUTPUT_DIR / "prompt_only_vs_rag_comparison.csv", index=False)
    print("Prompt-only passes:", int(p.sum()), "/ 52   RAG passes:", int(r.sum()), "/ 52   RAG failures:", len(failed)); print(comp["outcome"].value_counts())
    return failed, comp

def auto_triage_failures(failed):
    """Preliminary triage ONLY. A human must still check every row against the official KB/catalog."""
    rows = []
    for _, r in failed.iterrows():
        gold = _gold_sections(r["gold_evidence"]); ret = [x for x in r["retrieved_sections"].split(";") if x]; hit = any(g in ret for g in gold)
        greet = str(r.get("generic_greeting", "")).lower() == "true"; prod = "products.csv" in r["gold_evidence"]; phit = str(r.get("product_lookup_hit", "")).lower() == "true"
        if not gold and not prod: cat, note = "gold_evidence_missing", "No gold evidence in dataset; ask dataset owner."
        elif prod and not phit: cat, note = "product_lookup_failure", "Gold needs a catalog row but no product was looked up."
        elif gold and not hit: cat, note = "retrieval_failure", "No gold section ID in retrieved sections."
        elif greet: cat, note = "answer_generation_failure", "Gold retrieved but reply is a generic greeting."
        else: cat, note = "incorrect_or_incomplete_answer", "Gold retrieved; reply failed phrase check - read it; may be evaluation_phrase_mismatch."
        rows.append({"id": r["id"], "question": r["question"], "answer": r["answer"], "gold_evidence": r["gold_evidence"], "retrieved_sections": r["retrieved_sections"],
                     "failure_category_preliminary": cat, "review_notes": note, "review_status": "Preliminary automated triage; verify against official KB/catalog"})
    cols = ["id", "question", "answer", "gold_evidence", "retrieved_sections", "failure_category_preliminary", "review_notes", "review_status"]
    out = pd.DataFrame(rows, columns=cols); out.to_csv(OUTPUT_DIR / "manual_failure_review_rerun_triage.csv", index=False, encoding="utf-8-sig"); return out

def evaluate_conversations(limit_conversations=None):
    df = pd.read_csv(PRACTICE_DIR / "conversations.csv", dtype=str, keep_default_na=False)
    df["turn_num"] = pd.to_numeric(df["turn"], errors="coerce"); df = df.sort_values(["conversation_id", "turn_num"], kind="stable")
    if limit_conversations is not None: df = df[df.conversation_id.isin(list(df.conversation_id.drop_duplicates())[:limit_conversations])]
    if TICKETS_FILE.exists(): TICKETS_FILE.unlink()          # only tickets from THIS replay count
    agent = ZentroMartAgentLocal(use_rag=True); rows = []; last = None
    for _, r in df.iterrows():
        cid = r["conversation_id"]
        if cid != last: agent.reset(); last = cid
        answer = agent.respond(r["customer_message"])
        rows.append({"conversation_id": cid, "turn": r["turn"], "customer_message": r["customer_message"], "agent_reply": answer,
                     "expected_decision": r.get("expected_decision", ""), "expected_tool_calls": r.get("expected_tool_calls", ""), "gold_evidence": r.get("gold_evidence", ""),
                     "retrieved_sections": ";".join(x["section_id"] for x in agent.last_sources), "tool_calls_observed": agent.last_tools, "guard_triggered": agent.last_guard})
    res = pd.DataFrame(rows); res.to_csv(OUTPUT_DIR / "conversation_replay.csv", index=False, encoding="utf-8-sig")
    print("Saved", len(res), "turns,", res.conversation_id.nunique(), "conversations"); return res

def conversation_auto_flags(df):
    """Objective red flags to speed up manual review. These do NOT prove a turn passed."""
    tickets = set(pd.read_csv(TICKETS_FILE, dtype=str)["ticket_id"]) if TICKETS_FILE.exists() else set()
    f = pd.DataFrame({"conversation_id": df.conversation_id, "turn": df.turn})
    rep = df.agent_reply
    f["generic_greeting"] = [is_generic_greeting(a) and not GREETING_ONLY.match(m.strip()) for a, m in zip(rep, df.customer_message)]
    f["stall_phrase"] = rep.map(lambda a: bool(STALL.search(a))); f["placeholder"] = rep.map(lambda a: bool(PLACEHOLDER.search(a)))
    claimed = rep.map(lambda a: re.findall(r"\bTKT[0-9A-F]{6,}\b", a))
    f["ticket_claim_without_real_ticket"] = [any(t not in tickets for t in c) or (bool(TICKET_CLAIM.search(a)) and not c) for c, a in zip(claimed, rep)]
    f["escalation_expected_but_no_ticket_tool_call"] = [(d == "escalate") and ("create_ticket" not in t) for d, t in zip(df.expected_decision, df.tool_calls_observed)]
    f["lookup_expected_but_not_called"] = [("lookup_order" in e) and ("lookup_order" not in t) for e, t in zip(df.expected_tool_calls, df.tool_calls_observed)]
    print(f.drop(columns=["conversation_id", "turn"]).sum()); return f

def conversation_review_skeleton(df, flags):
    out = df[["conversation_id", "turn", "customer_message", "agent_reply", "expected_decision", "expected_tool_calls", "gold_evidence", "retrieved_sections", "tool_calls_observed"]].copy()
    out["auto_flags"] = ["; ".join(c for c in flags.columns[2:] if flags.loc[i, c]) for i in flags.index]
    out["manual_review_status"] = "Not manually validated"; out["manual_issue_category"] = ""; out["review_notes"] = ""
    out.to_csv(OUTPUT_DIR / "conversation_failure_review_rerun_template.csv", index=False, encoding="utf-8-sig"); return out

In [14]:
import tempfile

class _StubTok:
    def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=True):
        return "\n".join(m["content"] for m in messages)
class StubGen:
    """Stand-in for the LLM so tests are fast and deterministic."""
    def __init__(self, outputs=("Returns are accepted within the window shown in the policy.",)):
        self.outputs = list(outputs); self.calls = 0; self.prompts = []; self.tokenizer = _StubTok()
    def __call__(self, prompt, **kw):
        self.prompts.append(prompt); o = self.outputs[min(self.calls, len(self.outputs) - 1)]; self.calls += 1
        return [{"generated_text": o}]

def _mini_kb(tmp):
    kb = Path(tmp) / "kb"; kb.mkdir()
    (kb / "01_returns_policy.md").write_text('---\ndoc_id: P1\ntitle: "Returns Policy"\ndoc_type: policy\neffective_from: 2026-01-01\nstatus: current\n---\n## [RET-2] Return window\nElectronics can be returned within 7 days of delivery for a refund.\n', encoding="utf-8")
    (kb / "02_help_faq.md").write_text('---\ndoc_id: F1\ntitle: "Help FAQ"\ndoc_type: faq\neffective_from: 2026-01-01\nstatus: current\n---\n## [FAQ-1] Return window\nElectronics can be returned within 30 days of delivery for a refund.\n', encoding="utf-8")
    (kb / "03_old_returns.md").write_text('---\ndoc_id: P0\ntitle: "Old Returns Policy"\ndoc_type: policy\neffective_from: 2025-01-01\nstatus: superseded\n---\n## [RET-OLD] Return window\nElectronics can be returned within 3 days of delivery for a refund.\n', encoding="utf-8")
    return kb

def run_unit_tests():
    results = []
    def check(name, cond): results.append((name, bool(cond))); print(("PASS  " if cond else "FAIL  ") + name)
    tmp = tempfile.mkdtemp(); tf = Path(tmp) / "t.csv"; kb = _mini_kb(tmp)
    mini = Retriever(kb_dir=kb, use_dense=False)
    order = next(iter(ORDERS.values())); oid = str(order["order_id"]); cust = CUSTOMERS[order["customer_id"]]
    phone4 = str(cust["phone"])[-4:]; pin = str(order["delivery_pincode"]); h = len(oid) // 2
    def agent(outs=None, rag=True): return ZentroMartAgentLocal(use_rag=rag, generator=StubGen(outs) if outs else StubGen(), retriever=mini, tickets_file=tf)

    # --- number / date parsing
    check("Devanagari digits normalised", normalize_digits("७९३६ 3550") == "7936 3550")
    a = agent(); a.respond(f"Order number hai {oid[:h]} {oid[h:]}.")
    check("order id with a space is captured", a.pending_order_id == oid)
    a = agent(); a.respond(f"order {oid}"); before = a.pending_order_id; a.respond("Pincode 440010 hai")
    check("6-digit pincode is NOT taken as a new order id", a.pending_order_id == before == oid)
    a = agent(); a.respond(f"order {oid}"); a.respond("mera refund 17 October 2026 ko shuru hua tha")
    check("a year/date number is NOT used as verification", a.verification_attempts == 0 and a.verified_order is None)
    a = agent(); a.respond(f"order {oid}"); a.respond(f"I ordered 2 items, last four digits {phone4}")
    check("unrelated digits are not concatenated into the code", a.verified_order is not None)
    check("dates parsed (explicit + relative)", extract_dates("delivered 20 July")[0] == date(2026, 7, 20) and extract_dates("parso deliver hua")[0] == REFERENCE_DATE - timedelta(days=2))
    sun = date(2026, 10, 17)  # Saturday
    bd = next_business_days(sun, 3, {date(2026, 10, 19)})
    check("business days skip Sunday and holidays", bd == [date(2026, 10, 20), date(2026, 10, 21), date(2026, 10, 22)])
    check("query expansion adds English terms for Hindi text", "refund" in expand_query("रिफ़ंड कब आएगा").lower())

    # --- verification safety
    a = agent(); a.respond(f"order {oid}"); r = a.respond("Last four digits 0000" if phone4 != "0000" else "Last four digits 1111")
    check("wrong value does not reveal order data", a.verified_order is None and "status" not in r.lower() and pin not in r)
    r2 = a.respond("Last four digits 9999" if phone4 != "9999" else "Last four digits 8888"); r3 = a.respond(f"Last four digits {phone4}")
    check("order locks after 2 failures (no brute force)", a.verified_order is None and "unable to verify" in r3.lower())
    a = agent(); a.respond(f"order {oid}"); r = a.respond(f"Last four digits {phone4}")
    check("correct value verifies and reply uses tool result", a.verified_order is not None and oid in r and "lookup_order" in a.last_tools)
    g = a.generator; a.respond("what is the return window for electronics?")
    p = g.prompts[-1]
    check("model prompt never contains pincode/phone secrets", "delivery_pincode" not in p and str(cust["phone"]) not in p)
    check("verified order data persists to later turns", "VERIFIED ORDER DATA" in p)
    a = agent(); r = a.respond("Where is my order?")
    check("order question without verification asks for order number, no LLM call", a.generator.calls == 0 and "order number" in r.lower())
    a = agent(); a.respond(f"order {oid}"); r = a.respond("Please cancel it")
    check("order action before verification asks for verification, no LLM call", a.generator.calls == 0 and "last 4 digits" in r.lower())
    a = agent(); r = a.respond("Hi, I need some help with an order.")
    check("order-help opener asks for order number (not a generic greeting)", "order number" in r.lower() and not is_generic_greeting(r))

    # --- escalation / tickets
    a = agent(); r = a.respond("Actually, can I just speak to a person?"); m = re.search(r"TKT[0-9A-F]+", r)
    check("escalation creates a real ticket and quotes its ID", bool(m) and m.group() in tf.read_text())
    real = globals()["create_ticket"]; globals()["create_ticket"] = lambda *x, **k: {"created": False, "error": "down"}
    try: a = agent(); r = a.respond("I want to talk to a human agent"); ok = ("TKT" not in r) and "couldn't confirm" in r.lower()
    finally: globals()["create_ticket"] = real
    check("no ticket claim when the ticket tool fails", ok)
    a = agent(["I've created a ticket TKT123ABC456 for you. Returns take 7 days."]); r = a.respond("What is the return window for electronics?")
    check("model-invented ticket claims are removed", "TKT123" not in r and "7 days" in r)
    a = agent(); r = a.respond("Main is hafte teesri baar call kar rahi hoon isi baare mein.")
    check("repeat-contact escalates", "TKT" in a.last_tools or "ticket" in a.last_tools.lower())

    # --- output guards
    a = agent(["Hello! I'm ZentroMart's AI assistant. How can I help you today?", "Electronics can be returned within 7 days of delivery."])
    r = a.respond("What is the return window for electronics?")
    check("generic greeting is rejected and regenerated", a.generator.calls == 2 and "7 days" in r)
    a = agent(["How can I help you today?"]); r = a.respond("What is the return window for electronics?")
    check("persistent greeting falls back safely", not is_generic_greeting(r.replace("human colleague", "")) or "colleague" in r)
    a = agent(["According to RET-2, electronics can be returned within 7 days [RET-2]."]); r = a.respond("Return window for electronics?")
    check("internal section IDs are stripped", "RET-2" not in r and "7 days" in r)
    a = agent(["Your order was packed on [Date of packing].", "Electronics can be returned within 7 days."]); r = a.respond("Return window for electronics?")
    check("template placeholders are rejected", "[" not in r)
    a = agent(["I'm verifying your identity. Please wait a moment.", "Electronics can be returned within 7 days."]); r = a.respond("Return window for electronics?")
    check("stall phrases are rejected", "wait" not in r.lower())

    # --- retrieval / precedence / product lookup
    hits = mini.search("What is the return window for electronics?", k=3)
    ids = [x["section_id"] for x in hits]
    check("superseded document is excluded from the index", "RET-OLD" not in ids)
    check("current policy outranks FAQ on the same topic", ids and ids[0] == "RET-2")
    if CATALOG.rows and CATALOG.name_col:
        row = CATALOG.rows[0]
        check("product lookup by SKU", CATALOG.find(f"warranty for {row['sku']}")[0]["sku"] == row["sku"])
        check("product lookup by name", any(r["sku"] == row["sku"] for r in CATALOG.find(f"can I return the {row[CATALOG.name_col]}?")))
    a = agent(); a.respond("Return window for electronics?")
    check("follow-up retrieval query reuses previous question", "electronics" in a._retrieval_query("And for refund?", []).lower())
    failed = [n for n, ok in results if not ok]
    print(f"\n{len(results) - len(failed)}/{len(results)} unit tests passed" + (f"  FAILED: {failed}" if failed else ""))
    return results

In [15]:
TEST_RESULTS = run_unit_tests()
assert all(ok for _, ok in TEST_RESULTS), "Fix failing unit tests before running the evaluation."

PASS  Devanagari digits normalised
PASS  order id with a space is captured
PASS  6-digit pincode is NOT taken as a new order id
PASS  a year/date number is NOT used as verification
PASS  unrelated digits are not concatenated into the code
PASS  dates parsed (explicit + relative)
PASS  business days skip Sunday and holidays
PASS  query expansion adds English terms for Hindi text
PASS  wrong value does not reveal order data
PASS  order locks after 2 failures (no brute force)
PASS  correct value verifies and reply uses tool result
PASS  model prompt never contains pincode/phone secrets
PASS  verified order data persists to later turns
PASS  order question without verification asks for order number, no LLM call
PASS  order action before verification asks for verification, no LLM call
PASS  order-help opener asks for order number (not a generic greeting)
PASS  escalation creates a real ticket and quotes its ID
PASS  no ticket claim when the ticket tool fails
PASS  model-invented ticket clai

In [16]:
archive_original_run()
baseline_df = evaluate_single_turn(use_rag=False, limit=None)
rag_df = evaluate_single_turn(use_rag=True, limit=None)
s1 = save_evaluation(baseline_df, "step1_prompt_only_final")
s2 = save_evaluation(rag_df, "step2_rag_final")
failed_rag, comparison = build_failed_and_comparison()
(OUTPUT_DIR / "evaluation_summary.json").write_text(json.dumps({"dataset_questions": 52, "metric": "phrase-based", "step1": s1, "step2": s2}, indent=2), encoding="utf-8")
triage = auto_triage_failures(failed_rag); print(triage.failure_category_preliminary.value_counts())
display(failed_rag[["id", "question", "answer", "gold_evidence", "retrieved_sections", "must_include_pass", "must_not_include_pass"]].head(10))

No previous result files found to archive (upload the old ones into /content/evaluation_outputs first if you want to keep them).


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


step1_prompt_only_final {'n': 52, 'simple_pass': 6, 'simple_pass_rate': 0.11538461538461539, 'must_include_pass': 6, 'must_not_include_pass': 52, 'metric': 'phrase-based (must_include / must_not_include substring checks); not a human judgement of correctness'}
step2_rag_final {'n': 52, 'simple_pass': 21, 'simple_pass_rate': 0.40384615384615385, 'must_include_pass': 21, 'must_not_include_pass': 52, 'metric': 'phrase-based (must_include / must_not_include substring checks); not a human judgement of correctness'}
Prompt-only passes: 6 / 52   RAG passes: 21 / 52   RAG failures: 31
outcome
Failed both    31
Improved       15
Passed both     6
Name: count, dtype: int64
failure_category_preliminary
incorrect_or_incomplete_answer    19
gold_evidence_missing              4
retrieval_failure                  4
product_lookup_failure             4
Name: count, dtype: int64


,id,question,answer,gold_evidence,retrieved_sections,must_include_pass,must_not_include_pass
2,PQ0004,Can I schedule a delivery at midnight?,"Hi, I'm ZentroMart's AI assistant. Unfortunate...",,DEL-1;DEL-2;CAN-3;CAN-2;DEL-6;PAY-2,False,True
3,PQ0005,SKU ELE-0002 pe ZentroMart Assured hai kya? Cl...,"Namaste, main ZentroMart ka AI assistant hoon....",WAR-2 ; products.csv:ELE-0002,WAR-2;MKS-3;WAR-1;WAR-3;RET-1;DWM-4,False,True
5,PQ0008,credit card wala refund 17 October 2026 ko shu...,"Namaste, main ZentroMart ka AI assistant hoon....",REF-2 ; KB-00.3,REF-6;REF-2;RET-8;RET-7;REF-4;PAY-1,False,True
7,PQ0010,Is the collection free if the UrbanThread Slim...,"Hi, I'm ZentroMart's AI assistant. Based on th...",RET-6 ; DWM-2,DWM-4;WAR-3;DWM-1;DWM-2;RET-5;RET-4,False,True
8,PQ0013,"मैंने Gulmohar Matte Lipstick खोल लिया है, क्य...","नमस्ते, मैं ZentroMart का AI असिस्टेंट हूँ। गु...",RET-4,RET-2;RET-9;RET-3;RET-1;RET-5;DWM-4,False,True
10,PQ0015,Bulk order pe GST invoice milega kya?,"Namaste, main ZentroMart ka AI assistant hoon....",,MKS-1;IDV-2;CAN-1;REF-4;TRK-1;IDV-3,False,True
11,PQ0016,I opened the Neelam Eau de Parfum 50 ml and tr...,"Hi, I'm ZentroMart's AI assistant. The return ...",RET-4 ; RET-2,RET-2;RET-3;DWM-4;RET-9;RET-1;PLS-2,False,True
13,PQ0019,My Orbix Fit 3 Pro Smartwatch was delivered on...,"Hi, I'm ZentroMart's AI assistant. The return ...",RET-3 ; RETv3-2,RET-3;RET-2;RET-9;DWM-4;RET-6;PLS-2,False,True
14,PQ0020,मुझे Sootra Canvas Tote Bag लौटाने के लिए कितन...,"नमस्ते, मैं ZentroMart का AI असिस्टेंट हूँ। सू...",RET-2 ; RET-8 ; products.csv:FSH-0038,RET-3;RET-2;DWM-4;RET-9;RET-5;RET-1,False,True
15,PQ0021,What happens if I'm not home when the courier ...,"Hi, I'm ZentroMart's AI assistant. The courier...",DEL-3,DEL-3;PAY-3;REF-5;REF-1;TRK-1;TRK-2,False,True


In [17]:
conversation_df = evaluate_conversations()
conv_flags = conversation_auto_flags(conversation_df)
conv_template = conversation_review_skeleton(conversation_df, conv_flags)
display(conversation_df[["conversation_id", "turn", "customer_message", "agent_reply", "expected_decision", "tool_calls_observed"]].head(15))

Saved 96 turns, 23 conversations
generic_greeting                               0
stall_phrase                                   0
placeholder                                    0
ticket_claim_without_real_ticket               0
escalation_expected_but_no_ticket_tool_call    1
lookup_expected_but_not_called                 0
dtype: int64


,conversation_id,turn,customer_message,agent_reply,expected_decision,tool_calls_observed
0,PC002,1,"Hello, mujhe order ke baare mein help chahiye.","Namaste, main ZentroMart ka AI assistant hoon....",clarify,
1,PC002,2,Order number hai 8978 2837.,"Dhanyavaad. Aapki suraksha ke liye, kripya is ...",clarify,lookup_order(order_id=89782837) -> verificatio...
2,PC002,3,Last four digits 4533 hain.,"Dhanyavaad, aapka verification ho gaya hai. Or...",answer,"lookup_order(order_id=89782837, verification_v..."
3,PC002,4,Maine Zentro Vista 43-inch 4K Smart TV damage ...,I've created a support ticket (TKT7C6836BAB83D...,escalate,create_ticket() -> created=True ticket_id=TKT7...
4,PC003,1,"Hello, mujhe order ke baare mein help chahiye.","Namaste, main ZentroMart ka AI assistant hoon....",clarify,
5,PC003,2,Order number hai 1859 9253.,"Dhanyavaad. Aapki suraksha ke liye, kripya is ...",clarify,lookup_order(order_id=18599253) -> verificatio...
6,PC003,3,Last four digits 4265 hain.,"Dhanyavaad, aapka verification ho gaya hai. Or...",answer,"lookup_order(order_id=18599253, verification_v..."
7,PC003,4,Kya main Kaafi Running Sneakers return kar sak...,"Madam, aapki order 18599253 ka delivery 12 Oct...",answer,
8,PC003,5,Aur refund kitne din mein aayega?,"Madam, aapki order 18599253 ka delivery 12 Oct...",answer,
9,PC004,1,"नमस्ते, मुझे अपने ऑर्डर के बारे में मदद चाहिए।","नमस्ते, मैं ZentroMart का AI असिस्टेंट हूँ। ज़...",clarify,


In [18]:
t_order = next(iter(ORDERS.values())); t_id = str(t_order["order_id"])
un = lookup_order(t_id); assert un.get("verification_required") is True and "items" not in un
t_phone = str(CUSTOMERS[t_order["customer_id"]]["phone"])[-4:]
assert lookup_order(t_id, t_phone).get("verified") is True
assert lookup_order(t_id, "0000" if t_phone != "0000" else "1111").get("verified") is False
tt = create_ticket("Automated test: customer requested human review.", reason="automated_test", tickets_file=Path(tempfile.mkdtemp()) / "t.csv")
assert tt.get("created") is True
print("Unverified / verified / wrong-value / ticket tool tests: PASS")

Unverified / verified / wrong-value / ticket tool tests: PASS


In [19]:
ROOT_CAUSES = [
 "Spaced order numbers (e.g. '8978 2837') and Devanagari digits were never recognised, so verification could not start; digit groups are now parsed as standalone numbers and Unicode digits are normalised.",
 "Verification used ALL digits in a message concatenated together (dates/years/quantities could be mistaken for codes); now only standalone 4/6-digit groups are accepted, dates are excluded, and an order locks after 2 failed attempts (4 per conversation).",
 "The model prompt included delivery pincode / phone / customer_id from verified orders; these fields are now removed, and verified order data is kept for later turns.",
 "Retrieved context for the CURRENT turn was attached to the FIRST turn of the history in multi-turn chats; each turn now carries its own evidence and history stays clean.",
 "English-only embeddings (all-MiniLM-L6-v2) retrieved only FAQH chunks for Hindi/Hinglish queries; now a multilingual embedding model + TF-IDF hybrid (reciprocal-rank fusion) + cross-lingual query hints is used. Follow-up questions reuse the previous question for retrieval.",
 "Source precedence was left to similarity; now superseded / not-yet-effective documents are excluded and results are re-weighted by explicit authority (policy > agent guide > help/FAQ > marketing > logs).",
 "Product questions relied on policy embeddings; products.csv is now looked up by SKU or product name and the row (and its category terms) is added to the prompt and retrieval query.",
 "Refund/return date questions: relative dates are resolved against the 20 Oct 2026 reference date and a code-computed business-day (Mon-Sat, minus holidays) / calendar-day table is supplied instead of asking a 2B model to do date arithmetic.",
 "Generic greeting / 'checking, please wait' / unfilled placeholder / leaked section-ID replies are detected, regenerated once with a stricter prompt, then replaced by a safe fallback.",
 "Escalation relied on a few keywords and appended a ticket sentence regardless; tickets are now created only for defined escalation reasons, the reply quotes the ID only when create_ticket returns created=True, and model-invented ticket claims are removed.",
]
LIMITATIONS = [
 "The pass rate is PHRASE-BASED (must_include / must_not_include substring checks). It is not a human judgement of correctness; semantically correct answers can fail and vice versa.",
 "A 2B local model (gemma-2-2b-it) can still misread evidence, miscompute, or answer in the wrong language; Hindi/Hinglish quality is weaker than English.",
 "Holiday dates are auto-detected from the dataset files; the printed list must be checked by a person against the KB calendar.",
 "Day-counting convention for business days (day 1 = first business day after the event) is assumed unless the policy text says otherwise.",
 "Orders delivered before a policy version's effective date may need the older version; superseded documents are currently excluded, so such cases need manual review.",
 "Escalation rules cover human requests, repeat contacts and damage-approval requests only; other cases that policy says need human review rely on the model's answer and create no ticket.",
 "Tool calls in the replay are recorded from this code's own logs; production logs were not available.",
 "Manual reviews (failed cases and conversation turns) must be re-done for THIS rerun; reviews of the earlier run do not apply to new outputs.",
]

def write_report():
    s1, s2 = summarize_csv(OUTPUT_DIR / "step1_prompt_only_final.csv"), summarize_csv(OUTPUT_DIR / "step2_rag_final.csv")
    comp = pd.read_csv(OUTPUT_DIR / "prompt_only_vs_rag_comparison.csv", dtype=str)
    conv = pd.read_csv(OUTPUT_DIR / "conversation_replay.csv", dtype=str, keep_default_na=False)
    flags = conversation_auto_flags(conv)
    L = ["# Deliverable 1 - Evaluation Report", "", f"Reference date: {REFERENCE_DATE}. LLM: `{MODEL_ID}`. Embeddings: `{EMBED_MODEL}` + TF-IDF (RRF).", "",
         "## Metric", "The score is **phrase-based**: a question passes when every `must_include` phrase (any listed alternative) appears in the answer and no `must_not_include` phrase appears. It does not measure whether the answer is truly correct.", "",
         "## Single-turn results (52 questions)", "| Configuration | Passed | Rate |", "|---|---|---|",
         f"| Prompt-only | {s1['simple_pass']}/{s1['n']} | {s1['simple_pass_rate']:.1%} |", f"| RAG + tools (this rerun) | {s2['simple_pass']}/{s2['n']} | {s2['simple_pass_rate']:.1%} |", "",
         "Outcome of the per-question comparison: " + ", ".join(f"{k}: {v}" for k, v in comp["outcome"].value_counts().items()) + ".", ""]
    tri = OUTPUT_DIR / "manual_failure_review_rerun_triage.csv"
    if tri.exists() and len(pd.read_csv(tri)): L += ["Automatic triage of the remaining RAG failures (preliminary): " + ", ".join(f"{k}: {v}" for k, v in pd.read_csv(tri)["failure_category_preliminary"].value_counts().items()) + ".", ""]
    L += ["## Root causes fixed"] + [f"{i+1}. {t}" for i, t in enumerate(ROOT_CAUSES)] + ["", "Unit tests: " + (f"{sum(ok for _, ok in TEST_RESULTS)}/{len(TEST_RESULTS)} passed." if 'TEST_RESULTS' in globals() else "not run in this session."), ""]
    L += ["## Conversation replay", f"{len(conv)} turns across {conv.conversation_id.nunique()} conversations were replayed. Automatic red-flag counts (not proof of passing): " +
          ", ".join(f"{c}: {int(flags[c].sum())}" for c in flags.columns[2:]) + ".", ""]
    rv = OUTPUT_DIR / "conversation_failure_review.csv"
    if rv.exists():
        r = pd.read_csv(rv, dtype=str, keep_default_na=False); st = r["manual_review_status"].str.lower().value_counts().to_dict()
        L += ["Manual review status counts: " + ", ".join(f"{k}: {v}" for k, v in st.items()) + ".", ""]
    else:
        L += ["Manual conversation review for this rerun has NOT been completed; no claim is made that all turns pass.", ""]
    L += ["## Remaining limitations"] + [f"- {t}" for t in LIMITATIONS] + ["", "No API keys, tokens or `.env` files are included in the deliverable."]
    (OUTPUT_DIR / "evaluation_report.md").write_text("\n".join(L), encoding="utf-8"); print("\n".join(L))

write_report()

generic_greeting                               0
stall_phrase                                   0
placeholder                                    0
ticket_claim_without_real_ticket               0
escalation_expected_but_no_ticket_tool_call    1
lookup_expected_but_not_called                 0
dtype: int64
# Deliverable 1 - Evaluation Report

Reference date: 2026-10-20. LLM: `google/gemma-2-2b-it`. Embeddings: `paraphrase-multilingual-MiniLM-L12-v2` + TF-IDF (RRF).

## Metric
The score is **phrase-based**: a question passes when every `must_include` phrase (any listed alternative) appears in the answer and no `must_not_include` phrase appears. It does not measure whether the answer is truly correct.

## Single-turn results (52 questions)
| Configuration | Passed | Rate |
|---|---|---|
| Prompt-only | 6/52 | 11.5% |
| RAG + tools (this rerun) | 21/52 | 40.4% |

Outcome of the per-question comparison: Failed both: 31, Improved: 15, Passed both: 6.

Automatic triage of the remaining RAG fa

In [20]:
def secret_scan(paths):
    pats = [re.compile(r"hf_[A-Za-z0-9]{20,}"), re.compile(r"sk-[A-Za-z0-9_\-]{20,}"), re.compile(r"GNANI_API_KEY\s*=\s*\S+")]
    key = os.getenv("GNANI_API_KEY", ""); bad = []
    for p in paths:
        if p.suffix in (".csv", ".json", ".md", ".txt"):
            t = p.read_text(encoding="utf-8", errors="ignore")
            if any(pt.search(t) for pt in pats) or (key and key in t): bad.append(p.name)
        if p.name.startswith(".env") and p.name != ".env.example": bad.append(p.name)
    return bad

def final_checks():
    res = []
    def chk(name, ok, hard=True): res.append((name, ok, hard)); print(("PASS  " if ok else ("FAIL  " if hard else "WARN  ")) + name)
    f1, f2 = OUTPUT_DIR / "step1_prompt_only_final.csv", OUTPUT_DIR / "step2_rag_final.csv"
    d1, d2 = pd.read_csv(f1, dtype=str), pd.read_csv(f2, dtype=str)
    chk("both result CSVs have 52 rows", len(d1) == len(d2) == 52); chk("question IDs match", d1["id"].tolist() == d2["id"].tolist())
    for name, f in (("step1_prompt_only_final", f1), ("step2_rag_final", f2)):
        js = json.loads((OUTPUT_DIR / f"{name}_summary.json").read_text()); cs = summarize_csv(f)
        chk(f"{name} summary JSON agrees with CSV", all(js[k] == cs[k] for k in ("n", "simple_pass", "must_include_pass", "must_not_include_pass")))
    fc = pd.read_csv(OUTPUT_DIR / "step2_failed_cases_final.csv", dtype=str)
    chk("failed-case CSV matches latest RAG results", set(fc["id"]) == set(d2.loc[d2["simple_pass"].str.lower() != "true", "id"]))
    chk("failed-case & comparison files are newer than the RAG CSV", (OUTPUT_DIR / "step2_failed_cases_final.csv").stat().st_mtime >= f2.stat().st_mtime and (OUTPUT_DIR / "prompt_only_vs_rag_comparison.csv").stat().st_mtime >= f2.stat().st_mtime)
    comp = pd.read_csv(OUTPUT_DIR / "prompt_only_vs_rag_comparison.csv", dtype=str)
    chk("comparison covers all 52 questions", len(comp) == 52 and comp["id"].tolist() == d2["id"].tolist())
    chk("conversation replay exists", (OUTPUT_DIR / "conversation_replay.csv").exists())
    mf = OUTPUT_DIR / "manual_failure_review.csv"
    chk("manual failure review completed for THIS run (no 'Preliminary' rows)", mf.exists() and not pd.read_csv(mf, dtype=str)["review_status"].str.contains("Preliminary", na=False).any(), hard=False)
    cr = OUTPUT_DIR / "conversation_failure_review.csv"
    chk("conversation review has statuses and notes", cr.exists() and not pd.read_csv(cr, dtype=str)["manual_review_status"].str.contains("Not manually", na=False).any(), hard=False)
    chk("report exists and says the metric is phrase-based", (OUTPUT_DIR / "evaluation_report.md").exists() and "phrase-based" in (OUTPUT_DIR / "evaluation_report.md").read_text().lower())
    chk("no secrets in outputs", not secret_scan([p for p in OUTPUT_DIR.rglob("*") if p.is_file()]))
    hard = [n for n, ok, h in res if not ok and h]; print("\nHARD FAILURES:" if hard else "\nAll hard checks passed.", hard or "")
    return res

CHECKS = final_checks()

PASS  both result CSVs have 52 rows
PASS  question IDs match
PASS  step1_prompt_only_final summary JSON agrees with CSV
PASS  step2_rag_final summary JSON agrees with CSV
PASS  failed-case CSV matches latest RAG results
PASS  failed-case & comparison files are newer than the RAG CSV
PASS  comparison covers all 52 questions
PASS  conversation replay exists
WARN  manual failure review completed for THIS run (no 'Preliminary' rows)
WARN  conversation review has statuses and notes
PASS  report exists and says the metric is phrase-based
PASS  no secrets in outputs

All hard checks passed. 


In [21]:
wanted = ["step1_prompt_only_final.csv", "step2_rag_final.csv", "conversation_replay.csv", "step1_prompt_only_final_summary.json", "step2_rag_final_summary.json",
          "evaluation_summary.json", "prompt_only_vs_rag_comparison.csv", "step2_failed_cases_final.csv", "manual_failure_review.csv", "manual_failure_review_rerun_triage.csv",
          "conversation_failure_review.csv", "conversation_failure_review_rerun_template.csv", "evaluation_report.md"]
bad = secret_scan([OUTPUT_DIR / n for n in wanted if (OUTPUT_DIR / n).exists()]); assert not bad, f"Secrets found in {bad}; not packaging."
zip_out = OUTPUT_DIR.parent / "deliverable1_evaluation_files.zip"
with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as z:
    for n in wanted:
        p = OUTPUT_DIR / n
        if p.exists(): z.write(p, arcname=n); print("Included:", n)
        else: print("Not found (add it if required):", n)
print("ZIP:", zip_out)

Included: step1_prompt_only_final.csv
Included: step2_rag_final.csv
Included: conversation_replay.csv
Included: step1_prompt_only_final_summary.json
Included: step2_rag_final_summary.json
Included: evaluation_summary.json
Included: prompt_only_vs_rag_comparison.csv
Included: step2_failed_cases_final.csv
Not found (add it if required): manual_failure_review.csv
Included: manual_failure_review_rerun_triage.csv
Not found (add it if required): conversation_failure_review.csv
Included: conversation_failure_review_rerun_template.csv
Included: evaluation_report.md
ZIP: /content/deliverable1_evaluation_files.zip


In [23]:
%pip -q install gnani-vachana
try:
    from google.colab import userdata
    os.environ["GNANI_API_KEY"] = userdata.get("GNANI_API_KEY")
except Exception:
    pass
if os.getenv("GNANI_API_KEY"):
    from gnani.stt import GnaniSTTClient
    from gnani.tts import GnaniTTSClient
    stt, tts = GnaniSTTClient(), GnaniTTSClient()
    AUDIO_INPUT, AUDIO_OUTPUT = Path("/content/input_audio"), Path("/content/voice_test_outputs")
    AUDIO_INPUT.mkdir(exist_ok=True); AUDIO_OUTPUT.mkdir(exist_ok=True); manifest = []
    voice_agent = ZentroMartAgentLocal(use_rag=True)
    for ap in sorted(AUDIO_INPUT.iterdir()):
        if ap.suffix.lower() not in {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac"}: continue
        voice_agent.reset()
        heard = stt.transcribe(str(ap), language_code="en-IN")["transcript"]
        reply = voice_agent.respond(heard) if str(heard).strip() else "Sorry, I didn't catch that. Could you say it again?"
        out = AUDIO_OUTPUT / f"{ap.stem}_reply.wav"
        tts.synthesize(reply, voice="Kaveri", model="timbre-v2.5", language="en-IN", output_file=str(out))
        (AUDIO_OUTPUT / f"{ap.stem}_transcript.txt").write_text(heard, encoding="utf-8"); (AUDIO_OUTPUT / f"{ap.stem}_reply.txt").write_text(reply, encoding="utf-8")
        (AUDIO_OUTPUT / f"{ap.stem}_sources.json").write_text(json.dumps(voice_agent.last_sources, ensure_ascii=False, indent=2), encoding="utf-8")
        manifest.append({"input_audio": ap.name, "transcript": heard, "reply": reply, "reply_audio": out.name})
    (AUDIO_OUTPUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"); print("Processed audio files:", len(manifest))
else:
    print("GNANI_API_KEY not set - skipping the optional voice test.")

INFO  gnani | [STT REST] transcribe start | request_id=req_448705a2b779 | language=en-IN
INFO:gnani:[STT REST] transcribe start | request_id=req_448705a2b779 | language=en-IN
INFO  gnani | [STT REST] transcribe complete | request_id=req_448705a2b779
INFO:gnani:[STT REST] transcribe complete | request_id=req_448705a2b779
INFO  gnani | [TTS REST] synthesize start | request_id=req_98acbca1bbf5 | model=timbre-v2.5 | voice=Kaveri
INFO:gnani:[TTS REST] synthesize start | request_id=req_98acbca1bbf5 | model=timbre-v2.5 | voice=Kaveri
INFO  gnani | [TTS REST] synthesize complete | request_id=req_98acbca1bbf5
INFO:gnani:[TTS REST] synthesize complete | request_id=req_98acbca1bbf5


Processed audio files: 1
